In [1]:
df = spark.sql("SELECT * FROM bing_lake_db.tbl_latest_news")
display(df)

StatementMeta(, 6f23294f-a36c-42ec-bf8a-23a952ab9330, 3, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 0c655f58-2e96-4f61-8874-f3cc91ddd6ea)

In [2]:
import synapse.ml.core
from synapse.ml.services import AnalyzeText

StatementMeta(, 6f23294f-a36c-42ec-bf8a-23a952ab9330, 4, Finished, Available, Finished)

In [3]:
model=(AnalyzeText()
      .setTextCol("description")
      .setKind("SentimentAnalysis")
      .setOutputCol("response")
      .setErrorCol("error"))

StatementMeta(, 6f23294f-a36c-42ec-bf8a-23a952ab9330, 5, Finished, Available, Finished)

In [4]:
result=model.transform(df)

StatementMeta(, 6f23294f-a36c-42ec-bf8a-23a952ab9330, 6, Finished, Available, Finished)

In [5]:
display(result)

StatementMeta(, 6f23294f-a36c-42ec-bf8a-23a952ab9330, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 7830c7bb-30fc-4ed9-bf66-c62b53706f64)

In [6]:
from pyspark.sql.functions import col
sentiment_df= result.withColumn("sentiment",col("response.documents.sentiment"))


StatementMeta(, 6f23294f-a36c-42ec-bf8a-23a952ab9330, 8, Finished, Available, Finished)

In [7]:
display(sentiment_df)

StatementMeta(, 6f23294f-a36c-42ec-bf8a-23a952ab9330, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 47455f62-0012-4011-b1b6-dda12eac1141)

In [8]:
from pyspark.sql.utils import AnalysisException

try:
    table_name="tbl_sentiment_news"
    sentiment_df.write.format("delta").saveAsTable(table_name)

except AnalysisException:

    print("Table already exist")
    sentiment_df.createOrReplaceTempView("vw_sentiment_df")
    spark.sql(f""" MERGE INTO {table_name} as target_table
                   USING vw_sentiment_df AS source_table
                   ON source_table.url=target_table.url

                   WHEN MATCHED AND
                   source_table.title<>target_table.title OR
                   source_table.description<>target_table.description OR
                   source_table.category<>target_table.category OR
                   source_table.image<>target_table.image OR
                   source_table.provider<>target_table.provider OR
                   source_table.datePublished<>target_table.datePublished 

                   THEN UPDATE SET *

                   WHEN NOT MATCHED THEN INSERT *
             """)

StatementMeta(, 6f23294f-a36c-42ec-bf8a-23a952ab9330, 10, Finished, Available, Finished)

Table already exist


In [9]:
%%sql
select count(*) from bing_lake_db.tbl_sentiment_news

StatementMeta(, 6f23294f-a36c-42ec-bf8a-23a952ab9330, 11, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 1 fields>